In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.signal import savgol_filter
from scipy import stats
import yaml
plt.rcParams["animation.html"] = "jshtml"

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.gridspec import GridSpec
import tardigrade_functions as tg
from sklearn import metrics

In [ ]:
# settings for dataset
date = 20251209
home = os.path.expanduser("~")
expID = 'LB016'
inpath = os.path.join(home, f'data/{expID}/out_{date}')
pose_path = os.path.join(home, f'data/{expID}/poseestimation')
pose_file = 'DLC_Resnet50_TardiLocoOct23shuffle1_snapshot_200.csv'
cluster_path = os.path.join(inpath, 'cluster', 'out_cluster_050226', 'legs0-8')

In [ ]:
# load config
config_path = "config.yaml"
config = yaml.safe_load(open(config_path, "r"))

fps = config['video']['fps']

bodyparts = config['analysis']['bodyparts']
limbs = config['analysis']['limbs']
limbs_ord = config['analysis']['limbs_ord']
pairs = config['analysis']['pairs']
bp_pairs = config['analysis']['bp_pairs']

bp_color_dict = config['color']['bp_color']
roi_defined = config['analysis']['roi_defined']

idx = pd.IndexSlice

### Example image

In [ ]:
# load for one recording data
f = 'LB016043'
swing_bouts = pd.read_csv(os.path.join(inpath, f, f'{f}_swing.csv'), index_col=0)
stage = pd.read_csv(os.path.join(inpath, f, f'{f}_stage.csv'), index_col=0)
CLbp = pd.read_csv(os.path.join(inpath, f, f'{f}_CLbp.csv'), index_col=0, header=[0,1,2])
turns = pd.read_csv(os.path.join(inpath, f, f'{f}_turns.csv'), index_col=0)
#turns = np.hstack([[turns[0]],turns[1:][np.diff(turns)>1]])
label = pd.read_csv(os.path.join(cluster_path, f'{f}_gait.csv'))
label[label<0] = np.nan

In [ ]:
# rescale centerline body part position for plot
CLbp_norm = (CLbp - CLbp.mean(axis=0)) / CLbp.std(axis=0)
CLbp_x = CLbp_norm.loc[:,idx[:,:,'x']].droplevel(level=[0,2],axis=1).fillna(0)
CLbp_bg = CLbp_x.rolling(fps, min_periods=1, center=True).mean()

CLbp_x_smooth = pd.DataFrame(savgol_filter((CLbp_x-CLbp_bg).T, 15, 2).T)
CLbp_x_smooth.columns = CLbp_x.columns
CLbp_forimg = -((CLbp_x_smooth[limbs]-np.min(CLbp_x_smooth[limbs],axis=0))/(np.max(CLbp_x_smooth[limbs],axis=0) - np.min(CLbp_x_smooth[limbs],axis=0)))

In [ ]:
# plot example image

# set start ROI
roi = 500

# create layout for fig
fig = plt.figure(layout="constrained", figsize=(15,10))
gs = GridSpec(30,1, figure=fig)
arow = fig.add_subplot(gs[0, :])
cbar = fig.add_subplot(gs[1, :], sharex=arow)
hild = fig.add_subplot(gs[2:10, :], sharex=arow)
gait = fig.add_subplot(gs[11, :], sharex=arow)
trck = fig.add_subplot(gs[12:, :])

# set arrow to idicate turns
rng = (roi*fps, (roi+60)*fps-1)
arow.quiver(turns, np.ones(len(turns)), np.zeros(len(turns)), np.full(len(turns), -1), 
           angles='xy', scale_units='xy', units='xy', scale=1/1, width=1.5)
arow.set_ylim(0,1)
arow.set_yticks([])
arow.set_ylabel('turns', rotation=0, labelpad=5, loc='top')
arow.set_frame_on(False)

# plot colorbar corresponding to time
cbar_forimg = np.zeros(len(CLbp_x_smooth))
cbar_forimg[rng[0]:rng[1]] = np.arange(rng[1]-rng[0])
cbar.imshow(cbar_forimg[np.newaxis], rasterized=True, aspect='auto', cmap='plasma')
cbar.set_xticks(np.arange(*rng,fps*5))
cbar.set_xticklabels(np.arange(*np.divide(rng,fps).astype(int),5))
cbar.set_yticks([])
cbar.set_ylabel('colored position\non track', rotation=0, labelpad=5, loc='top')
arow.set_frame_on(False)

# plot the gait with hildebrandt style
hild.imshow(swing_bouts[limbs].astype(int).T,interpolation='none',rasterized=True, cmap='Greys_r')
for bp in limbs:
    hild.plot((CLbp_forimg+[.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5])[bp].values, c=bp_color_dict[bp], alpha=.5)
hild.set_yticks(range(len(limbs)))
hild.set_yticklabels(limbs)
hild.axis('auto')

# plot gait type / cluster used
gait.imshow(label.T, rasterized=True, aspect='auto')
gait.set_xlim(*rng)
gait.set_yticks([])
gait.set_ylabel('cluster', rotation=0, labelpad=5, loc='top')
gait.set_frame_on(False)

# plot cms track
trck.plot(stage.loc[:,'Xcms'],stage.loc[:,'Ycms'],) #complete track
trck.scatter(*stage.loc[rng[0]:rng[1]-1,['Xcms','Ycms']].T.values, s=150, c=np.arange(*rng), cmap='plasma') # corresponding to shown roi
# arrows to sho turns
trck_quiv = [t for t in turns.values if t in np.arange(*rng)]
trck_quiv = np.concat(trck_quiv) if trck_quiv else trck_quiv       
trck.quiver(*stage.loc[trck_quiv,['Xcms','Ycms']].T.values, np.zeros(len(trck_quiv)), np.full(len(trck_quiv),1), 
            angles='xy', scale_units='xy', units='xy', scale=1/200, width=20, pivot='tip')
# add scale bar
scale = 1000
scalebar = AnchoredSizeBar(trck.transData,
                            scale, f'{scale} um', 'lower left', 
                            pad=-0.5,
                            color='black',
                            frameon=False,
                            size_vertical=2,)
trck.add_artist(scalebar)
trck.axis('equal')
trck.axis('off')

plt.savefig(os.path.join(cluster_path, f'{f}_longexample.pdf'), bbox_inches='tight')
plt.show()

### differences in likelihood

In [ ]:
common_all = pd.DataFrame([])
for f in os.listdir(inpath):
    if expID in f and '.' not in f:
        label = pd.read_csv(os.path.join(cluster_path, f'{f}_gait.csv'), header=None)
        label[label.values<0] = np.nan
        label.columns = ['label']

        likeli = pd.read_csv(os.path.join(pose_path,f+pose_file), index_col=0, header=[0,1,2]).loc[:,idx[:,:,'likelihood']].mean(axis=1)
        likeli.name = 'likelihood'
        
        straight = pd.read_csv(os.path.join(inpath,f,f'{f}_straight.csv'), index_col=0,).astype(int)
        turns = pd.read_csv(os.path.join(inpath, f, f'{f}_turns.csv'), index_col=0).values
        turns = np.concat([np.arange(t-7,t+8) for t in turns.flatten()])
        straight.iloc[turns] = 2
        straight.columns = ['straight']

        common_f = pd.concat([label, likeli, straight], axis=1)
        common_f.index = pd.MultiIndex.from_product([[f],common_f.index])
        common_all = pd.concat([common_all, common_f])

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(7,3), sharey=True, width_ratios=(4,3))
for name, gr in common_all.groupby('label'):
    if name < 0:
        continue
    axs[0].boxplot(gr['likelihood'].dropna(), positions=[name], showfliers=False, widths=.8)
    axs[0].set_xlabel('label')
    axs[0].set_ylabel('likelihood')

for name, gr in common_all.groupby('straight'):
    axs[1].boxplot(gr['likelihood'].dropna(), positions=[name], showfliers=False, widths=.8)
    axs[1].set_xticks([0,1,2])
    axs[1].set_xticklabels(['bend', 'straight', 'turns'])
    axs[1].set_xlabel('track curvature')


plt.savefig(os.path.join(cluster_path, f'likelihood_grouped.pdf'), bbox_inches='tight')


### Comparison of automatic and manual

In [ ]:
# load manual annotation
man_gait = pd.read_csv('C:/Users/boeger/data/LB016/test/manual_labeling.csv', header=[0,1,2,3]) - 1 # because manual labeling done in fiji#.values
man_ids = man_gait.columns.get_level_values(0).unique()
man_on = man_gait.loc[0,idx[:,'frame']].values.astype(int)
man_dict = {r: man_gait.loc[:,idx[r,:,:,['on','off']]].values.reshape(len(man_gait), 2, 8).swapaxes(1,2).swapaxes(0,1) for r in man_ids} #bp #time #on/off

In [ ]:
# load automatic annotataion and compare with manual
comarr_list = []
for i,f in enumerate(man_ids):
    swing_bouts = pd.read_csv(os.path.join(inpath, f, f'{f}_swing.csv'), index_col=0).astype(int)
    swing_dict = tg.get_bouts(swing_bouts[man_on[i]:int(man_on[i]+(11*fps))])
    aut_arr = pd.concat([pd.DataFrame(swing_dict[k]) for k in swing_dict], axis=1).values.reshape(-1,8,2).swapaxes(0,1)
    aut_arr = np.cumsum(aut_arr, axis=-1)

    def closest(ar1, iter):
        if any(np.isnan(ar1[iter])):
            return -1
        dist = abs(ar2[~np.any(np.isnan(ar2),axis=1)] - ar1[iter])
        return np.argmin(dist, axis=0)[0]

    def find(iter_range):
        return closest(ar1, iter_range)

    common_arr = np.full((2, 8, man_dict[f].shape[1]+aut_arr.shape[1], 2),np.nan) #shape: man/aut, legs, swing events, on/off

    for bp in range(man_dict[f].shape[0]):
        ar1 = man_dict[f][bp]
        ar2 = aut_arr[bp]
        # find matching index of ar2 to ar1, by smallest absolute distance of value in ar2 to values ar1
        matched_idx = np.array(list(map(find, range(ar1.shape[0]))))
        # populate common_arr
        common_arr[0,bp,:ar1.shape[0]] = ar1
        common_arr[1,bp,:ar1.shape[0]] = ar2[matched_idx]
        # set 2nd negative if index repeats 
        common_arr[1,bp,np.where(np.diff(matched_idx)==0)] = -(common_arr[1,bp,np.where(np.diff(matched_idx)==0)])

        # repeat with values in ar2 (aut_arr) that have not been matched
        lost_aut = [i for i in range(ar2.shape[0]) if i not in matched_idx]
        # swap roles
        ar2 = man_dict[f][bp]
        ar1 = aut_arr[bp][lost_aut]
        lost_matched_idx = np.array(list(map(find, range(ar1.shape[0])))).astype(int)
        # insert into common_arr
        # set negative, because is a repeated index for ar2
        common_arr[0,bp,len(matched_idx):len(matched_idx)+len(ar1)] = -ar2[lost_matched_idx]
        common_arr[1,bp,len(matched_idx):len(matched_idx)+len(ar1)] = ar1
    
    comarr_list.append(common_arr)

In [ ]:
# calculate pearson r for all values in comarr_list
comarr_flat = np.vstack([common_arr[~np.isnan(common_arr)].reshape(-1,2) for common_arr in comarr_list])
comarr_r = stats.pearsonr(*comarr_flat.T, alternative='greater')
np.square(comarr_r), comarr_r, comarr_flat.shape

In [ ]:
# plot pairs
for common_arr in comarr_list:
    plt.scatter(common_arr[0,:,:,0],common_arr[1,:,:,0], c='k', label='onset', alpha=0.3)
    plt.scatter(common_arr[0,:,:,1],common_arr[1,:,:,1], c='grey', label='offset', alpha=0.3)
    c= 0
plt.axvline(0,c='k')
plt.axhline(0,c='k')
plt.xlabel('manual')
plt.ylabel('automatic')
plt.legend(bbox_to_anchor=(0,0),loc='lower left')
plt.axis('equal')

plt.savefig(os.path.join(cluster_path, f'corr_onoff_matched.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# calculate distance between automatic and manual on and offsets
common_dist = []
for common_arr in comarr_list:
    common_ = common_arr.reshape(2,-1,2)
    common_ = abs(common_[1]) - abs(common_[0])
    common_dist.append(common_)
common_dist = np.concat(common_dist)
common_dist = common_dist[~np.any(np.isnan(common_dist), axis=1)]

In [ ]:
# calculate pearson r for distance
comdist_r = stats.pearsonr(*common_dist.T, alternative='greater')
np.square(comdist_r), comdist_r, common_dist.shape

In [ ]:
# 10 and 90 percentiles
np.percentile(common_dist, 10), np.percentile(common_dist, 90)

In [ ]:
# plot 2d histogram for on and offset pairs
Z, xedges, yedges = np.histogram2d(*common_dist.T, bins=30, range=[[-15,15],[-15,15]])
plt.pcolormesh(xedges, yedges, Z.T, cmap='magma')
plt.xticks([-fps,0,fps],[-1,0,1])
plt.yticks([-fps,0,fps],[-1,0,1])
plt.xlabel('onset predicted relative to manual (s)')
plt.ylabel('offset predicted relative to manual (s)')
plt.axvline(0,c='w', ls=':')
plt.axhline(0,c='w', ls=':')
plt.colorbar()
plt.axis('equal')
plt.savefig(os.path.join(cluster_path, f'corr_onoff_diff.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# load automatic according to man_on and transform manual into ethogram
auto_etho = []
man_etho = []
clbp_img = []
for i,f in enumerate(man_ids):
    print(f)
    swing_bouts = pd.read_csv(os.path.join(inpath, f, f'{f}_swing.csv'), index_col=0).astype(int)

    CLbp = pd.read_csv(os.path.join(inpath, f, f'{f}_CLbp.csv'), index_col=0, header=[0,1,2])
    CLbp_norm = (CLbp - CLbp.mean(axis=0)) / CLbp.std(axis=0)
    CLbp_x = CLbp_norm.loc[:,idx[:,:,'x']].droplevel(level=[0,2],axis=1).fillna(0)

    auto_ = swing_bouts[man_on[i]:man_on[i]+(10*fps)].T
    clbp_ = ((CLbp_x[limbs]-np.min(CLbp_x[limbs],axis=0))/(np.max(CLbp_x[limbs],axis=0) - np.min(CLbp_x[limbs],axis=0))).iloc[man_on[i]:man_on[i]+(10*fps)]
    
    man_ = np.zeros_like(auto_)
    for bp in range(man_.shape[0]):
        on = man_dict[f][bp,:,0]
        on = on[~np.isnan(on)]
        off = man_dict[f][bp,:,1]
        off = off[~np.isnan(off)]
        etho = tg.ethogram_fromOnOff(on,off, arr_len=12*15)
        man_[bp] = etho[:10*fps]
    man_etho.append(man_)
    auto_etho.append(auto_)
    clbp_img.append(clbp_)
    

In [ ]:
# calculate performance metrics
man_etho_flat = np.concat(man_etho).flatten()
auto_etho_flat = np.concat(auto_etho).flatten()
print(metrics.accuracy_score(man_etho_flat, auto_etho_flat),
      metrics.precision_score(man_etho_flat, auto_etho_flat),
      metrics.recall_score(man_etho_flat, auto_etho_flat),
      metrics.confusion_matrix(man_etho_flat, auto_etho_flat))

In [ ]:
# plot manual, automatic and overlap hildebrandt style gait
for i,f in enumerate(man_ids):
    fig, axs = plt.subplots(3, sharex=True, figsize=(10,6))
    axs[0].imshow(man_etho[i], interpolation='None', aspect='auto', cmap='Greys_r')
    axs[1].imshow(auto_etho[i], interpolation='None', aspect='auto', cmap='Greys_r')
    axs[2].imshow(man_etho[i], interpolation='None', aspect='auto', cmap='Greys_r')
    axs[2].imshow(auto_etho[i], interpolation='None', aspect='auto', cmap='Greys_r', alpha=.5)
    for bp in limbs:
        for ax in axs:
            ax.plot((-clbp_img[i][limbs]+[.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5])[bp].values, c=bp_color_dict[bp], alpha=.5)
        
    plt.savefig(os.path.join(cluster_path, f'corr_{f}_hild.pdf'), bbox_inches='tight')